In [8]:
# Install if needed
#!pip install rdflib
#!pip install networkx

In [9]:
# Import von rdflib, networkx und andere libraries
from rdflib import Graph, Namespace, RDF
import networkx as nx
from collections import deque
import matplotlib.pyplot as plt

In [10]:
# === 4. BFS with class filtering ===
def bfs_with_class_filter(G, start_node, target_class_uri):
    visited = set()
    queue = deque([start_node])
    matched_nodes = []

    while queue:
        node = queue.popleft()
        if node in visited:
            continue
        visited.add(node)

        if G.nodes[node].get('class') == target_class_uri:
            matched_nodes.append(node)

        for neighbor in G.neighbors(node):
            if neighbor not in visited:
                queue.append(neighbor)

    return matched_nodes

In [11]:
# Gerichtete Graph erstellen
def graph_from_rdf(g):
    
    G = nx.DiGraph()
    
    # Extract rdf:type information
    types = {}
    for s, p, o in g:
        if p == RDF.type:
            types[str(s)] = str(o)
    
    # Add all triples except rdf:type as edges
    for s, p, o in g:
        s_str, o_str, p_str = str(s), str(o), str(p)
        G.add_edge(s_str, o_str, label=p_str)
    
    # Add node classes as attributes
    for node in G.nodes:
        G.nodes[node]['class'] = types.get(node, None)

    # === 3. Show the entire KG ===
    print("Knowledge Graph:")
    for u, v, d in G.edges(data=True):
        label = d['label'].split("/")[-1]
        print(f"{u.split('/')[-1]} --[{label}]--> {v.split('/')[-1]}")

    
    return G

In [12]:
# SCHRITT 1

g = Graph()

# Namespace definieren
SC = Namespace("http://example.org/")

# Triples
g.add((SC.supplyChain1, RDF.type, SC.SupplyChain))

g.add((SC.hersteller,            RDF.type, SC.Firma))
g.add((SC.teilzulieferer,        RDF.type, SC.Firma))
g.add((SC.einzelhaendler1,       RDF.type, SC.Firma))
g.add((SC.einzelhaendler2,       RDF.type, SC.Firma))

g.add((SC.bestand,               RDF.type, SC.ValueStream))
g.add((SC.produktion,            RDF.type, SC.ValueStream))
g.add((SC.verkauf1,              RDF.type, SC.ValueStream))
g.add((SC.verkauf2,              RDF.type, SC.ValueStream))

g.add((SC.vorlaufzeitBestand,    RDF.type, SC.Vorlaufzeit))
g.add((SC.vorlaufzeitProduktion, RDF.type, SC.Vorlaufzeit))
g.add((SC.vorlaufzeitVerkauf1,   RDF.type, SC.Vorlaufzeit))
g.add((SC.vorlaufzeitVerkauf2,   RDF.type, SC.Vorlaufzeit))

g.add((SC.hersteller,            SC.istTeilVon, SC.supplyChain1))
g.add((SC.teilzulieferer,        SC.istTeilVon, SC.hersteller))
g.add((SC.einzelhaendler1,       SC.istTeilVon, SC.supplyChain1))
g.add((SC.einzelhaendler2,       SC.istTeilVon, SC.supplyChain1))

g.add((SC.bestand,               SC.istTeilVon, SC.teilzulieferer))
g.add((SC.produktion,            SC.istTeilVon, SC.hersteller))
g.add((SC.verkauf1,              SC.istTeilVon, SC.einzelhaendler1))
g.add((SC.verkauf2,              SC.istTeilVon, SC.einzelhaendler2))

<Graph identifier=N0cca306160024dc9a4a6d796a90b48e5 (<class 'rdflib.graph.Graph'>)>

In [13]:
# SCHRITT 2

G = graph_from_rdf(g)


Knowledge Graph:
teilzulieferer --[22-rdf-syntax-ns#type]--> Firma
teilzulieferer --[istTeilVon]--> hersteller
verkauf2 --[istTeilVon]--> einzelhaendler2
verkauf2 --[22-rdf-syntax-ns#type]--> ValueStream
einzelhaendler2 --[istTeilVon]--> supplyChain1
einzelhaendler2 --[22-rdf-syntax-ns#type]--> Firma
supplyChain1 --[22-rdf-syntax-ns#type]--> SupplyChain
bestand --[istTeilVon]--> teilzulieferer
bestand --[22-rdf-syntax-ns#type]--> ValueStream
vorlaufzeitBestand --[22-rdf-syntax-ns#type]--> Vorlaufzeit
vorlaufzeitVerkauf1 --[22-rdf-syntax-ns#type]--> Vorlaufzeit
vorlaufzeitProduktion --[22-rdf-syntax-ns#type]--> Vorlaufzeit
hersteller --[22-rdf-syntax-ns#type]--> Firma
hersteller --[istTeilVon]--> supplyChain1
vorlaufzeitVerkauf2 --[22-rdf-syntax-ns#type]--> Vorlaufzeit
produktion --[istTeilVon]--> hersteller
produktion --[22-rdf-syntax-ns#type]--> ValueStream
einzelhaendler1 --[istTeilVon]--> supplyChain1
einzelhaendler1 --[22-rdf-syntax-ns#type]--> Firma
verkauf1 --[22-rdf-syntax-ns#ty

In [14]:
# SCHRITT 3

start_node = str(SC.supplyChain1)
target_class = str(SC.ValueStream)

# Rückwärts traversal BFS
rwarts_G = G.reverse()

results = bfs_with_class_filter(rwarts_G, start_node, target_class)

print("Gefundene ValueStreams:")
for r in results:
    print(" -", r.split("/")[-1])

# Vorlaufzeiten aus dem Graphen summieren
gesamtzeit = 0
for u, v, d in G.edges(data=True):
    if d['label'].endswith('tage'):
        gesamtzeit += int(v.split("/")[-1])  # falls als Literal gespeichert

print(f"\nGesamte Vorlaufzeit: {gesamtzeit} Tage")

Gefundene ValueStreams:
 - verkauf2
 - produktion
 - verkauf1
 - bestand

Gesamte Vorlaufzeit: 0 Tage
